In [1]:
%matplotlib inline
import random
import torch
from d2l import torch as d2l

In [7]:
class SyntheticRegressionData (d2l.DataModule): #@save
    """Synthetic data for linear regression."""
    def __init__(self, w, b, noise=0.01, num_train=1000, num_val=1000, batch_size=32):
       super().__init__()
       self.save_hyperparameters()
       n = num_train + num_val
       self.X = torch.randn(n, len(w))
       noise = torch.randn(n, 1) * noise
       self.y = torch.matmul(self.X, w.reshape((-1,1))) + b + noise

In [8]:
data = SyntheticRegressionData(w=torch.tensor([2, -3.4]), b=4.2)
print('features:', data.X[0], '\nlabel:', data.y[0])

features: tensor([2.1453, 0.5966]) 
label: tensor([6.4890])


In [9]:
@d2l.add_to_class(SyntheticRegressionData)
def get_dataloader(self, train):
    if train:
        indices = list(range(0, self.num_train))
        #Shuffle them for training
        random.shuffle(indices)
    else:
        #Else get the range for the validation set
        indices = list(range(self.num_train, self.num_train + self.num_val))

    for i in range(0, len(indices), self.batch_size):
        #return the next batch of features and labels
        batch_indices = torch.tensor(indices[i: i + self.batch_size])
        yield self.X[batch_indices], self.y[batch_indices]

In [11]:
X, y = next(iter(data.get_dataloader(train=True)))
print('X shape:', X.shape, 'y shape:', y.shape)

X shape: torch.Size([32, 2]) y shape: torch.Size([32, 1])


In [12]:
#Using our data loader is inefficient, we can use the built in DataLoader from PyTorch
@d2l.add_to_class(d2l.DataModule) #@save
def get_tensorloader(self, tensors, train, indices=slice(0,None)):
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, self.batch_size, shuffle=train)

@d2l.add_to_class(SyntheticRegressionData) #@save
def get_dataloader(self, train):
    i = slice(0, self.num_train) if train else slice(self.num_train, None)
    return self.get_tensorloader((self.X, self.y), train, i)

In [13]:
X, y = next(iter(data.get_dataloader(train=True)))
print('X shape:', X.shape, 'y shape:', y.shape)

X shape: torch.Size([32, 2]) y shape: torch.Size([32, 1])


In [14]:
len(data.train_dataloader())

32